# Global carbon benefit raster by Forces of Nature mangrove patch

This notebook processes `G_CarbonBenefit_9s.tif` for Jamaica and associates it with the Forces of Nature mangrove patches.

Workflow:
1. Clip the global raster to Jamaica with an offshore buffer so coastal ecosystems are retained.
2. Reproject the clipped raster to Jamaica's metric grid (`EPSG:3448`).
3. Intersect the reprojected raster with Forces of Nature mangrove polygons using patch-level zonal statistics.
4. Fill any no-raster-coverage patches using nearest-neighbour observed mangrove patches.
5. Join positive avoided-EAD attribution fields so carbon and coastal-protection benefits can be analysed together by patch.

Unit assumption: raster values are treated as source carbon-benefit values per hectare. Patch totals are calculated as `sum(pixel values) × pixel area in hectares`. If the source documentation defines the raster differently, update the total-value calculation before reporting.

## Imports

In [ ]:
import os
import sys
from pathlib import Path

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
matplotlib_config_dir = base_path / ".matplotlib"
matplotlib_config_dir.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_config_dir))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from matplotlib.patches import Patch
from rasterio.enums import Resampling
from rasterio.errors import WindowError
from rasterio.features import geometry_mask, geometry_window
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject
from shapely.geometry import LineString

robyn_libraries_path = (base_path / "robyns_libraries").resolve()
if str(robyn_libraries_path) not in sys.path:
    sys.path.append(str(robyn_libraries_path))
import Robyn_paper_2_defs

pd.set_option("display.max_columns", 200)

## Paths and settings

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"
jamaica_clip_buffer_m = 20_000
carbon_benefit_units_assumed = "tonnes C per hectare"
include_all_touched_pixels = False

global_carbon_raster_path = base_path / "dphil_paper_3/inputs/carbon/global_carbon/G_CarbonBenefit_9s.tif"
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
forces_of_nature_mangrove_path = base_path / "dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp"
mangrove_priority_table_path = base_path / "dphil_paper_3/results_coastal_scenario_comparison/weighted_area_distance_signed/mangrove_priority_ranking/mangrove_priority_full_table_weighted_area_distance.csv"

output_dir = base_path / "dphil_paper_3/processed_data/carbon/global_carbon_benefit"
figure_dir = base_path / "dphil_paper_3/results/co_benefits/carbon/global_carbon_benefit"
output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

global_carbon_source_clip_path = output_dir / "global_carbon_benefit_jamaica_20km_buffer_epsg4326.tif"
global_carbon_projected_path = output_dir / "global_carbon_benefit_jamaica_20km_buffer_epsg3448.tif"
patch_summary_csv_path = output_dir / "fn_mangrove_patch_global_carbon_benefit_summary.csv"
patch_summary_gpkg_path = output_dir / "fn_mangrove_patch_global_carbon_benefit_summary.gpkg"
nearest_neighbor_summary_csv_path = output_dir / "fn_mangrove_patch_global_carbon_benefit_nearest_neighbor_fill.csv"
nearest_neighbor_summary_gpkg_path = output_dir / "fn_mangrove_patch_global_carbon_benefit_nearest_neighbor_fill.gpkg"
nearest_neighbor_links_gpkg_path = output_dir / "fn_mangrove_patch_global_carbon_benefit_nearest_neighbor_links.gpkg"
patch_table_path = output_dir / "fn_mangrove_patch_global_carbon_benefit_table.csv"
parish_summary_csv_path = output_dir / "fn_mangrove_patch_global_carbon_benefit_by_parish.csv"

patch_map_path = figure_dir / "fn_mangrove_patch_global_carbon_benefit_map.png"
no_coverage_map_path = figure_dir / "fn_mangrove_patch_global_carbon_benefit_no_valid_cells_map.png"
nearest_neighbor_source_map_path = figure_dir / "fn_mangrove_patch_global_carbon_benefit_nearest_neighbor_sources_map.png"

for required_path in [global_carbon_raster_path, jamaica_boundary_path, forces_of_nature_mangrove_path, mangrove_priority_table_path]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required input: {required_path}")

print(f"Global carbon raster: {global_carbon_raster_path}")
print(f"Jamaica boundary: {jamaica_boundary_path}")
print(f"Forces of Nature mangroves: {forces_of_nature_mangrove_path}")
print(f"Outputs: {output_dir}")

## Load Jamaica land polygon and Forces of Nature mangroves

In [ ]:
jamaica_boundary = gpd.read_file(jamaica_boundary_path).to_crs(jamaica_metric_grid_crs)
jamaica_boundary = jamaica_boundary[jamaica_boundary.geometry.notna() & ~jamaica_boundary.geometry.is_empty].copy()
jamaica_boundary["geometry"] = jamaica_boundary.geometry.make_valid()
jamaica_clip_area = gpd.GeoDataFrame(
    {"clip_buffer_m": [jamaica_clip_buffer_m]},
    geometry=[jamaica_boundary.geometry.union_all().buffer(jamaica_clip_buffer_m)],
    crs=jamaica_metric_grid_crs,
)

fn_mangroves = gpd.read_file(forces_of_nature_mangrove_path).to_crs(jamaica_metric_grid_crs)
fn_mangroves = fn_mangroves[fn_mangroves.geometry.notna() & ~fn_mangroves.geometry.is_empty].copy()
fn_mangroves["geometry"] = fn_mangroves.geometry.make_valid()

if fn_mangroves["ID"].isna().any():
    raise ValueError("The Forces of Nature mangrove layer has missing patch IDs in the ID field.")

fn_mangroves["Mangrove_ID"] = fn_mangroves["ID"].astype(int)
fn_mangroves["patch_area_m2"] = fn_mangroves.geometry.area
fn_mangroves["patch_area_ha"] = fn_mangroves["patch_area_m2"] / 10_000

print(f"Jamaica clip buffer: {jamaica_clip_buffer_m / 1000:.1f} km")
print(f"Mangrove patches: {len(fn_mangroves):,}")
display(fn_mangroves[["Mangrove_ID", "Parish", "TYPE", "patch_area_ha"]].head())

## Clip the global raster to Jamaica plus offshore buffer

The global raster is too large to reproject directly. This cell first clips it to a filled Jamaica land polygon buffered by 20 km in the Jamaica metric CRS, then saves the clipped raster in the source CRS (`EPSG:4326`). The filled land polygon is used deliberately so terrestrial carbon across Jamaica is retained; line-buffer boundary layers should not be used here because they create a ring with an empty interior.

In [ ]:
with rasterio.open(global_carbon_raster_path) as carbon_source:
    source_nodata = carbon_source.nodata
    if source_nodata is None:
        source_nodata = -9999.0

    clip_area_raster_crs = jamaica_clip_area.to_crs(carbon_source.crs)
    clipped_values, clipped_transform = mask(
        carbon_source,
        clip_area_raster_crs.geometry,
        crop=True,
        filled=True,
        nodata=source_nodata,
    )

    clipped_profile = carbon_source.profile.copy()
    clipped_profile.update(
        {
            "height": clipped_values.shape[1],
            "width": clipped_values.shape[2],
            "transform": clipped_transform,
            "nodata": source_nodata,
            "compress": "lzw",
        }
    )

    with rasterio.open(global_carbon_source_clip_path, "w", **clipped_profile) as clipped_destination:
        clipped_destination.write(clipped_values)

clipped_band = clipped_values[0]
valid_clip_pixels = np.isfinite(clipped_band) & (clipped_band != source_nodata)
clip_metadata = pd.DataFrame(
    [
        {"property": "source_crs", "value": str(clipped_profile["crs"])},
        {"property": "clip_shape", "value": f"{clipped_band.shape[0]:,} rows × {clipped_band.shape[1]:,} columns"},
        {"property": "valid_clip_pixels", "value": int(valid_clip_pixels.sum())},
        {"property": "clip_min", "value": float(clipped_band[valid_clip_pixels].min()) if valid_clip_pixels.any() else np.nan},
        {"property": "clip_mean", "value": float(clipped_band[valid_clip_pixels].mean()) if valid_clip_pixels.any() else np.nan},
        {"property": "clip_max", "value": float(clipped_band[valid_clip_pixels].max()) if valid_clip_pixels.any() else np.nan},
        {"property": "nodata", "value": source_nodata},
        {"property": "units_assumed", "value": carbon_benefit_units_assumed},
    ]
)
display(clip_metadata)
print(f"Saved clipped source-CRS raster: {global_carbon_source_clip_path}")

## Reproject clipped raster to Jamaica CRS

The clipped raster is reprojected to `EPSG:3448` so patch areas and pixel areas are in metres/hectares.

In [ ]:
with rasterio.open(global_carbon_source_clip_path) as carbon_clip:
    target_transform, target_width, target_height = calculate_default_transform(
        carbon_clip.crs,
        jamaica_metric_grid_crs,
        carbon_clip.width,
        carbon_clip.height,
        *carbon_clip.bounds,
    )
    pixel_area_ha = abs(target_transform.a * target_transform.e) / 10_000

    target_profile = carbon_clip.profile.copy()
    target_profile.update(
        {
            "crs": jamaica_metric_grid_crs,
            "transform": target_transform,
            "width": target_width,
            "height": target_height,
            "nodata": source_nodata,
            "compress": "lzw",
        }
    )

    with rasterio.open(global_carbon_projected_path, "w", **target_profile) as carbon_destination:
        reproject(
            source=rasterio.band(carbon_clip, 1),
            destination=rasterio.band(carbon_destination, 1),
            src_transform=carbon_clip.transform,
            src_crs=carbon_clip.crs,
            src_nodata=source_nodata,
            dst_transform=target_transform,
            dst_crs=jamaica_metric_grid_crs,
            dst_nodata=source_nodata,
            resampling=Resampling.bilinear,
        )

reproject_metadata = pd.DataFrame(
    [
        {"property": "target_crs", "value": jamaica_metric_grid_crs},
        {"property": "target_shape", "value": f"{target_height:,} rows × {target_width:,} columns"},
        {"property": "target_pixel_area_ha", "value": pixel_area_ha},
        {"property": "target_resolution_m", "value": abs(target_transform.a)},
    ]
)
display(reproject_metadata)
print(f"Saved Jamaica CRS raster: {global_carbon_projected_path}")

## Calculate patch-level zonal statistics

In [ ]:
def calculate_patch_zonal_statistics(mangrove_patches, raster_path, nodata_value, all_touched):
    patch_rows = []

    with rasterio.open(raster_path) as carbon_raster:
        for mangrove_patch in mangrove_patches[["Mangrove_ID", "geometry"]].itertuples(index=False):
            try:
                patch_window = geometry_window(carbon_raster, [mangrove_patch.geometry])
            except WindowError:
                patch_rows.append(
                    {
                        "valid_carbon_benefit_pixel_count": 0,
                        "min_carbon_benefit_value": np.nan,
                        "mean_carbon_benefit_value": np.nan,
                        "max_carbon_benefit_value": np.nan,
                        "sum_carbon_benefit_values": np.nan,
                    }
                )
                continue

            raster_values = carbon_raster.read(1, window=patch_window, masked=False)
            patch_mask = geometry_mask(
                [mangrove_patch.geometry],
                out_shape=raster_values.shape,
                transform=carbon_raster.window_transform(patch_window),
                invert=True,
                all_touched=all_touched,
            )
            valid_pixel_mask = patch_mask & np.isfinite(raster_values)
            if nodata_value is not None:
                valid_pixel_mask = valid_pixel_mask & (raster_values != nodata_value)

            patch_values = raster_values[valid_pixel_mask]
            if patch_values.size == 0:
                patch_rows.append(
                    {
                        "valid_carbon_benefit_pixel_count": 0,
                        "min_carbon_benefit_value": np.nan,
                        "mean_carbon_benefit_value": np.nan,
                        "max_carbon_benefit_value": np.nan,
                        "sum_carbon_benefit_values": np.nan,
                    }
                )
            else:
                patch_rows.append(
                    {
                        "valid_carbon_benefit_pixel_count": int(patch_values.size),
                        "min_carbon_benefit_value": float(patch_values.min()),
                        "mean_carbon_benefit_value": float(patch_values.mean()),
                        "max_carbon_benefit_value": float(patch_values.max()),
                        "sum_carbon_benefit_values": float(patch_values.sum()),
                    }
                )

    return pd.DataFrame(patch_rows)


carbon_statistics = calculate_patch_zonal_statistics(
    fn_mangroves,
    global_carbon_projected_path,
    source_nodata,
    include_all_touched_pixels,
)

carbon_benefit_summary = fn_mangroves[
    [
        "Mangrove_ID",
        "Parish",
        "TYPE",
        "TARGET",
        "CONDITION",
        "GEOCLIMATI",
        "HECTARES",
        "patch_area_m2",
        "patch_area_ha",
        "geometry",
    ]
].join(carbon_statistics)

carbon_benefit_summary["valid_carbon_benefit_pixel_count"] = carbon_benefit_summary[
    "valid_carbon_benefit_pixel_count"
].fillna(0).astype(int)
carbon_benefit_summary["has_observed_carbon_benefit"] = carbon_benefit_summary[
    "valid_carbon_benefit_pixel_count"
] > 0
carbon_benefit_summary["carbon_benefit_status"] = np.where(
    carbon_benefit_summary["has_observed_carbon_benefit"],
    "Observed raster overlap",
    "No valid raster cells",
)
carbon_benefit_summary["coverage_area_ha"] = carbon_benefit_summary["valid_carbon_benefit_pixel_count"] * pixel_area_ha
carbon_benefit_summary["coverage_pct_of_patch"] = np.where(
    carbon_benefit_summary["patch_area_ha"] > 0,
    carbon_benefit_summary["coverage_area_ha"] / carbon_benefit_summary["patch_area_ha"] * 100,
    np.nan,
)
# The raster value is interpreted as tonnes C/ha. Pixel area and summed pixel values are retained only
# as diagnostics; carbon totals are density multiplied by the actual mangrove patch area.
carbon_benefit_summary["carbon_benefit_per_patch_ha"] = carbon_benefit_summary[
    "mean_carbon_benefit_value"
].where(carbon_benefit_summary["has_observed_carbon_benefit"])
carbon_benefit_summary["total_carbon_benefit_source_units"] = (
    carbon_benefit_summary["carbon_benefit_per_patch_ha"] * carbon_benefit_summary["patch_area_ha"]
)

display(
    carbon_benefit_summary.sort_values("total_carbon_benefit_source_units", ascending=False)[
        [
            "Mangrove_ID",
            "Parish",
            "TYPE",
            "patch_area_ha",
            "valid_carbon_benefit_pixel_count",
            "coverage_pct_of_patch",
            "mean_carbon_benefit_value",
            "total_carbon_benefit_source_units",
        ]
    ].head(20)
)

## Nearest-neighbour fill and avoided-EAD join

Patches with no valid raster cells are filled using the per-hectare carbon-benefit value from the nearest Forces of Nature mangrove patch with observed raster coverage. The transfer value is calculated as `nearest source patch total carbon benefit / nearest source patch area`, then multiplied by the missing patch area. The positive avoided-EAD values are then joined from the weighted mangrove priority output.

In [ ]:
observed_carbon_patches = carbon_benefit_summary[carbon_benefit_summary["has_observed_carbon_benefit"]].copy()
missing_carbon_patches = carbon_benefit_summary[~carbon_benefit_summary["has_observed_carbon_benefit"]].copy()

if observed_carbon_patches.empty:
    raise ValueError("No Forces of Nature mangrove patches have observed global carbon-benefit raster coverage.")

nearest_neighbor_rows = []
for missing_patch in missing_carbon_patches.itertuples(index=False):
    source_distances_m = observed_carbon_patches.geometry.distance(missing_patch.geometry)
    source_candidates = pd.DataFrame(
        {
            "source_index": source_distances_m.index,
            "nearest_source_distance_m": source_distances_m.to_numpy(),
            "nearest_source_mangrove_id": observed_carbon_patches["Mangrove_ID"].to_numpy(),
        }
    ).sort_values(["nearest_source_distance_m", "nearest_source_mangrove_id"])
    nearest_source = observed_carbon_patches.loc[source_candidates.iloc[0]["source_index"]]
    nearest_source_carbon_benefit_per_ha = nearest_source["carbon_benefit_per_patch_ha"]

    nearest_neighbor_rows.append(
        {
            "Mangrove_ID": missing_patch.Mangrove_ID,
            "nearest_source_mangrove_id": int(nearest_source["Mangrove_ID"]),
            "nearest_source_parish": nearest_source["Parish"],
            "nearest_source_type": nearest_source["TYPE"],
            "nearest_source_distance_m": float(source_candidates.iloc[0]["nearest_source_distance_m"]),
            "nearest_source_carbon_benefit_per_ha": float(nearest_source_carbon_benefit_per_ha),
            "nn_total_carbon_benefit_source_units": float(
                nearest_source_carbon_benefit_per_ha * missing_patch.patch_area_ha
            ),
        }
    )

nearest_neighbor_fill_columns = [
    "Mangrove_ID",
    "nearest_source_mangrove_id",
    "nearest_source_parish",
    "nearest_source_type",
    "nearest_source_distance_m",
    "nearest_source_carbon_benefit_per_ha",
    "nn_total_carbon_benefit_source_units",
]
nearest_neighbor_fill = pd.DataFrame(nearest_neighbor_rows, columns=nearest_neighbor_fill_columns)
carbon_benefit_summary = carbon_benefit_summary.merge(nearest_neighbor_fill, on="Mangrove_ID", how="left")
carbon_benefit_summary["nearest_source_distance_km"] = carbon_benefit_summary["nearest_source_distance_m"] / 1_000
carbon_benefit_summary["carbon_benefit_estimate_method"] = np.where(
    carbon_benefit_summary["has_observed_carbon_benefit"],
    "observed_raster_overlap",
    "nearest_observed_patch_mean",
)
carbon_benefit_summary["carbon_benefit_per_patch_ha_with_nn_fill"] = carbon_benefit_summary[
    "carbon_benefit_per_patch_ha"
].where(
    carbon_benefit_summary["has_observed_carbon_benefit"],
    carbon_benefit_summary["nearest_source_carbon_benefit_per_ha"],
)
carbon_benefit_summary["total_carbon_benefit_with_nn_fill"] = carbon_benefit_summary[
    "total_carbon_benefit_source_units"
].where(
    carbon_benefit_summary["has_observed_carbon_benefit"],
    carbon_benefit_summary["nn_total_carbon_benefit_source_units"],
)

nearest_neighbor_fill_gdf = carbon_benefit_summary[~carbon_benefit_summary["has_observed_carbon_benefit"]].copy()
nearest_neighbor_fill_output_columns = [
    "Mangrove_ID",
    "Parish",
    "TYPE",
    "patch_area_ha",
    "nearest_source_mangrove_id",
    "nearest_source_parish",
    "nearest_source_type",
    "nearest_source_distance_m",
    "nearest_source_distance_km",
    "nearest_source_carbon_benefit_per_ha",
    "nn_total_carbon_benefit_source_units",
]

nearest_neighbor_link_rows = []
if len(nearest_neighbor_fill_gdf) > 0:
    nearest_neighbor_source_lookup = observed_carbon_patches.set_index("Mangrove_ID")
    for filled_patch in nearest_neighbor_fill_gdf.itertuples(index=False):
        source_patch = nearest_neighbor_source_lookup.loc[int(filled_patch.nearest_source_mangrove_id)]
        nearest_neighbor_link_rows.append(
            {
                "Mangrove_ID": filled_patch.Mangrove_ID,
                "nearest_source_mangrove_id": int(filled_patch.nearest_source_mangrove_id),
                "nearest_source_distance_m": filled_patch.nearest_source_distance_m,
                "nearest_source_distance_km": filled_patch.nearest_source_distance_km,
                "geometry": LineString(
                    [
                        filled_patch.geometry.representative_point(),
                        source_patch.geometry.representative_point(),
                    ]
                ),
            }
        )
nearest_neighbor_links = gpd.GeoDataFrame(nearest_neighbor_link_rows, crs=carbon_benefit_summary.crs)

patch_positive_avoided_ead = pd.read_csv(
    mangrove_priority_table_path,
    usecols=["Mangrove_ID", "avoided_usd_min", "avoided_usd_max", "avoided_usd_mean"],
).rename(
    columns={
        "avoided_usd_min": "avoided_ead_usd_min",
        "avoided_usd_max": "avoided_ead_usd_max",
        "avoided_usd_mean": "patch_positive_avoided_EADs",
    }
)
carbon_benefit_summary = carbon_benefit_summary.merge(patch_positive_avoided_ead, on="Mangrove_ID", how="left")

observed_total_carbon_benefit = carbon_benefit_summary["total_carbon_benefit_source_units"].fillna(0)
nearest_neighbor_total_carbon_benefit = carbon_benefit_summary[
    "nn_total_carbon_benefit_source_units"
].where(~carbon_benefit_summary["has_observed_carbon_benefit"])
patch_table = pd.DataFrame(
    {
        "mangrove_patch_id": carbon_benefit_summary["Mangrove_ID"].astype(int),
        "patch_size_hectares": carbon_benefit_summary["patch_area_ha"],
        "global_carbon_benefit_value": observed_total_carbon_benefit,
        "global_carbon_benefit_value_per_hectare": np.where(
            carbon_benefit_summary["patch_area_ha"] > 0,
            observed_total_carbon_benefit / carbon_benefit_summary["patch_area_ha"],
            np.nan,
        ),
        "distance_to_nearest_neighbour_km_for_no_data": carbon_benefit_summary["nearest_source_distance_km"].where(
            ~carbon_benefit_summary["has_observed_carbon_benefit"]
        ),
        "carbon_benefit_from_nearest_neighbour": nearest_neighbor_total_carbon_benefit,
        "nearest_neighbour_source_carbon_benefit_per_hectare_used": carbon_benefit_summary[
            "nearest_source_carbon_benefit_per_ha"
        ].where(~carbon_benefit_summary["has_observed_carbon_benefit"]),
        "carbon_benefit_from_nearest_neighbour_per_hectare": np.where(
            carbon_benefit_summary["patch_area_ha"] > 0,
            nearest_neighbor_total_carbon_benefit / carbon_benefit_summary["patch_area_ha"],
            np.nan,
        ),
        "patch_positive_avoided_EADs": carbon_benefit_summary["patch_positive_avoided_EADs"],
        "avoided_ead_usd_min": carbon_benefit_summary["avoided_ead_usd_min"],
        "avoided_ead_usd_max": carbon_benefit_summary["avoided_ead_usd_max"],
    }
).sort_values("mangrove_patch_id")
patch_table_decimal_columns = [column for column in patch_table.columns if column != "mangrove_patch_id"]
patch_table[patch_table_decimal_columns] = patch_table[patch_table_decimal_columns].round(2)

method_summary = pd.DataFrame(
    [
        {"metric": "patches_with_observed_raster_coverage", "value": int(carbon_benefit_summary["has_observed_carbon_benefit"].sum())},
        {"metric": "patches_filled_by_nearest_neighbor", "value": len(nearest_neighbor_fill_gdf)},
        {"metric": "observed_total_carbon_benefit_tonnes_c", "value": carbon_benefit_summary["total_carbon_benefit_source_units"].sum(skipna=True)},
        {"metric": "nearest_neighbor_filled_total_carbon_benefit_tonnes_c", "value": carbon_benefit_summary["total_carbon_benefit_with_nn_fill"].sum(skipna=True)},
        {"metric": "carbon_density_method", "value": "Observed patches use mean raster tonnes C/ha; totals equal density multiplied by actual mangrove patch hectares."},
        {"metric": "nearest_neighbor_method", "value": "Missing patches inherit nearest observed patch tonnes C/ha; totals equal inherited density multiplied by actual mangrove patch hectares."},
    ]
)

display(method_summary)
display(patch_table.head(20))

## Summarise and save outputs

In [ ]:
parish_summary = (
    carbon_benefit_summary.groupby("Parish", dropna=False)
    .agg(
        patch_count=("Mangrove_ID", "count"),
        observed_patch_count=("has_observed_carbon_benefit", "sum"),
        patch_area_ha=("patch_area_ha", "sum"),
        total_carbon_benefit_source_units=("total_carbon_benefit_source_units", "sum"),
        total_carbon_benefit_with_nn_fill=("total_carbon_benefit_with_nn_fill", "sum"),
        mean_carbon_benefit_value=("mean_carbon_benefit_value", "mean"),
        mean_carbon_benefit_per_patch_ha_with_nn_fill=("carbon_benefit_per_patch_ha_with_nn_fill", "mean"),
        patch_positive_avoided_EADs=("patch_positive_avoided_EADs", "sum"),
        avoided_ead_usd_min=("avoided_ead_usd_min", "sum"),
        avoided_ead_usd_max=("avoided_ead_usd_max", "sum"),
    )
    .reset_index()
)
parish_summary["filled_patch_count"] = parish_summary["patch_count"] - parish_summary["observed_patch_count"]
parish_summary["carbon_benefit_per_patch_ha"] = np.where(
    parish_summary["patch_area_ha"] > 0,
    parish_summary["total_carbon_benefit_source_units"] / parish_summary["patch_area_ha"],
    np.nan,
)
parish_summary["carbon_benefit_per_patch_ha_with_nn_fill"] = np.where(
    parish_summary["patch_area_ha"] > 0,
    parish_summary["total_carbon_benefit_with_nn_fill"] / parish_summary["patch_area_ha"],
    np.nan,
)
parish_summary = parish_summary.sort_values("total_carbon_benefit_with_nn_fill", ascending=False)

patch_summary_columns = [column for column in carbon_benefit_summary.columns if column != "geometry"]
carbon_benefit_summary[patch_summary_columns].to_csv(patch_summary_csv_path, index=False)
carbon_benefit_summary.to_file(patch_summary_gpkg_path, driver="GPKG")
patch_table.to_csv(patch_table_path, index=False, float_format="%.2f")
nearest_neighbor_fill_gdf[nearest_neighbor_fill_output_columns].to_csv(nearest_neighbor_summary_csv_path, index=False)
if len(nearest_neighbor_fill_gdf) > 0:
    nearest_neighbor_fill_gdf.to_file(nearest_neighbor_summary_gpkg_path, driver="GPKG")
if len(nearest_neighbor_links) > 0:
    nearest_neighbor_links.to_file(nearest_neighbor_links_gpkg_path, driver="GPKG")
parish_summary.to_csv(parish_summary_csv_path, index=False)

display(parish_summary)
print(f"Saved patch CSV: {patch_summary_csv_path}")
print(f"Saved patch GeoPackage: {patch_summary_gpkg_path}")
print(f"Saved patch table: {patch_table_path}")
print(f"Saved nearest-neighbour CSV: {nearest_neighbor_summary_csv_path}")
print(f"Saved parish CSV: {parish_summary_csv_path}")

## Maps

In [ ]:
def style_jamaica_map(axis, title_text):
    axis.set_title(title_text)
    axis.set_axis_off()
    Robyn_paper_2_defs.draw_scale_bar(
        axis,
        location=(0.88, 0.78),
        length_km=20,
        linewidth=0.6,
        label_offset=0.02,
        km_offset=0.01,
    )
    Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)


figure, axis = plt.subplots(figsize=(8, 10))
carbon_benefit_summary.plot(
    ax=axis,
    column="total_carbon_benefit_source_units",
    cmap="YlGn",
    legend=True,
    legend_kwds={"shrink": 0.55, "label": "Total carbon benefit (tonnes C)"},
    linewidth=0.15,
    edgecolor="black",
    missing_kwds={"color": "lightgrey", "label": "No raster cells"},
)
style_jamaica_map(axis, "Global carbon benefit by Forces of Nature mangrove patch")
figure.tight_layout()
figure.savefig(patch_map_path, dpi=300, bbox_inches="tight")
plt.show()

no_coverage_patches = carbon_benefit_summary[~carbon_benefit_summary["has_observed_carbon_benefit"]].copy()
figure, axis = plt.subplots(figsize=(9, 10))
if len(no_coverage_patches) > 0:
    no_coverage_patches.plot(ax=axis, color="#C62828", edgecolor="black", linewidth=0.25)
    no_coverage_points = no_coverage_patches.set_geometry(no_coverage_patches.representative_point())
    no_coverage_points.plot(ax=axis, color="#C62828", edgecolor="white", linewidth=0.4, markersize=28, marker="o")
    for no_coverage_patch in no_coverage_points.itertuples(index=False):
        axis.annotate(
            str(int(no_coverage_patch.Mangrove_ID)),
            xy=(no_coverage_patch.geometry.x, no_coverage_patch.geometry.y),
            xytext=(3, 3),
            textcoords="offset points",
            fontsize=5,
            color="black",
        )
else:
    carbon_benefit_summary.plot(ax=axis, color="#BDBDBD", edgecolor="black", linewidth=0.15)
    axis.text(0.5, 0.5, "All patches have valid raster cells", transform=axis.transAxes, ha="center", va="center")
style_jamaica_map(axis, "FN mangrove patches with no valid global carbon-benefit cells")
axis.legend(
    handles=[Patch(facecolor="#C62828", edgecolor="black", label="No valid raster cells")],
    loc="lower left",
    frameon=True,
)
figure.tight_layout()
figure.savefig(no_coverage_map_path, dpi=300, bbox_inches="tight")
plt.show()

figure, axis = plt.subplots(figsize=(9, 10))
if len(nearest_neighbor_fill_gdf) > 0:
    nearest_source_ids = nearest_neighbor_fill_gdf["nearest_source_mangrove_id"].dropna().astype(int).unique()
    nearest_source_patches = carbon_benefit_summary[carbon_benefit_summary["Mangrove_ID"].isin(nearest_source_ids)].copy()
    nearest_source_points = nearest_source_patches.set_geometry(nearest_source_patches.representative_point())
    nearest_fill_points = nearest_neighbor_fill_gdf.set_geometry(nearest_neighbor_fill_gdf.representative_point())
    nearest_neighbor_links.plot(ax=axis, color="#757575", linewidth=0.35, alpha=0.7)
    nearest_source_patches.plot(ax=axis, color="#1565C0", edgecolor="black", linewidth=0.2, alpha=0.75)
    nearest_neighbor_fill_gdf.plot(ax=axis, color="#C62828", edgecolor="black", linewidth=0.25, alpha=0.9)
    nearest_source_points.plot(ax=axis, color="#1565C0", edgecolor="white", linewidth=0.3, markersize=20, marker="o")
    nearest_fill_points.plot(ax=axis, color="#C62828", edgecolor="white", linewidth=0.3, markersize=20, marker="o")
else:
    carbon_benefit_summary.plot(ax=axis, color="#BDBDBD", edgecolor="black", linewidth=0.15)
    axis.text(0.5, 0.5, "No nearest-neighbour fill required", transform=axis.transAxes, ha="center", va="center")
style_jamaica_map(axis, "Nearest observed carbon-benefit source patches for no-coverage mangroves")
source_link_legend_handles = [
    Patch(facecolor="#C62828", edgecolor="black", label="No valid cells / filled patch"),
    Patch(facecolor="#1565C0", edgecolor="black", label="Nearest observed source patch"),
    Patch(facecolor="#757575", edgecolor="#757575", label="Nearest-neighbour link"),
]
axis.legend(handles=source_link_legend_handles, loc="lower left", frameon=True)
figure.tight_layout()
figure.savefig(nearest_neighbor_source_map_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved map: {patch_map_path}")
print(f"Saved no-coverage map: {no_coverage_map_path}")
print(f"Saved nearest-neighbour source map: {nearest_neighbor_source_map_path}")

## How to use these outputs

- Use `Mangrove_ID` / `mangrove_patch_id` as the patch join key.
- Use `total_carbon_benefit_source_units` for observed raster-only patch totals; the column name is retained for compatibility, but values are tonnes C.
- Use `total_carbon_benefit_with_nn_fill` when no-coverage patches should be filled from nearest observed mangrove patches.
- Review `nearest_source_distance_km` and the nearest-neighbour source map before final reporting.
- Use `patch_positive_avoided_EADs`, `avoided_ead_usd_min`, and `avoided_ead_usd_max` to compare carbon benefit with coastal-protection benefit by patch.